## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### 1. Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("google_genai:gemini-2.5-flash-lite")
model

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x0000020693FB6660>, default_metadata=(), model_kwargs={})

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="Year of this movies was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movies rating out of 10")

In [12]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000206F4BCD7F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000206F4BCE270>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'Year of this movies was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out

In [6]:
response = model.invoke("Provide details about the moview Inception")
print(response.content)

<think>
Okay, I need to provide details about the movie "Inception". Let me start by recalling what I know. It's a sci-fi action film directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio. The title "Inception" refers to the concept of planting an idea into someone's mind, as opposed to stealing information, which is called extraction in the movie. 

The film's plot is a bit complex. There's a group of people who can enter people's dreams to influence their thoughts. The main character, Dom Cobb, is a thief who steals secrets by entering the subconscious of his targets. He's offered a chance to erase his criminal past by performing the reverse: planting an idea. The team needs to go through multiple layers of dreams, each deeper than the last. But there's a risk because if the dreamer's subconscious is too strong, they can get stuck in a limbo state, which is a place between the real world and the dream world.

The supporting cast includes Joseph Gordon-Levitt, Elle

#### Message output alongside parsed structure

In [13]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [23]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")

In [24]:
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that info. Inception was directed by Christopher Nolan, right? The title is "Inception", and it came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm those details. Yep, that\'s correct. So I\'ll structure the tool call with those parameters. Make sure the JSON is correctly formatted with the required fields. No need to include any extra information not specified in the function. Alright, ready to output the tool call.\n', 'tool_calls': [{'id': '1sy72bynm', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 195, 'prompt_tokens': 231, 't

#### Nested Structure

In [43]:
from numpy._core.defchararray import title
from openai.types.admin.organization import role
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role : str

class Moviedetails(BaseModel):
    title : str
    year : int
    cast : list[Actor]
    cost : float | None = Field(None, description="Budget in Crores")

model_with_structure = model.with_structured_output(Moviedetails)
response = model_with_structure.invoke("Provide details about the telugu movie EEGA")

In [44]:
response

Moviedetails(title='Eega', year=2012, cast=[Actor(name='Nani', role='Nani'), Actor(name='Sudeep', role='Sudeep'), Actor(name='Samantha Ruth Prabhu', role='Bindu')], cost=24.0)

### 2.TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [45]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie with details"""
    title:Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")

In [46]:
response

{'title': 'Avengers', 'year': 2012, 'director': 'Joss Whedon', 'rating': 8.0}

In [47]:
class Actor(TypedDict):
    name : str
    role : str

class Moviedetails(TypedDict):
    title : str
    year : int
    cast : list[Actor]
    cost : float | None = Field(None, description="Budget in Crores")

model_with_structure = model.with_structured_output(Moviedetails)
response = model_with_structure.invoke("Provide details about the telugu movie EEGA")

In [48]:
response

{'title': 'Eega',
 'year': 2012,
 'cast': [{'name': 'Nani', 'role': 'Sudeep'},
  {'name': 'Samantha Ruth Prabhu', 'role': 'Bindu'},
  {'name': 'Sudeep', 'role': 'Prince'}],
 'cost': 26000000.0}

In [49]:
model.profile

{'name': 'Gemini 2.5 Flash-Lite',
 'release_date': '2025-06-17',
 'last_updated': '2025-06-17',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

### 3. DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [56]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [51]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='17b5c174-de57-405c-816f-569670fe2a0c'),
  AIMessage(content='{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e8d86-29d5-72b3-8d58-84df2638fe3a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [52]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [54]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}